In [1]:
import cv2
import torch
import numpy as np
import onnxruntime as ort
import ipywidgets as widgets
from IPython.display import display
from jetcam.utils import bgr8_to_jpeg
from jetracer.nvidia_racecar import NvidiaRacecar
from jetcam.csi_camera import CSICamera
from torch2trt import TRTModule
from utils import preprocess

# 1. Hardware
car = NvidiaRacecar()
camera = CSICamera(width=224, height=224, capture_fps=30)
camera.running = True

# 2. Road Following Model (TensorRT - Twoja mocna strona)
model_road = TRTModule()
model_road.load_state_dict(torch.load('road_following_model_merged_0337t_tanh_norbert_laptop_trt.pth'))

# 3. YOLO Model (ONNX z akceleracją TensorRT)
providers = [
    ('TensorrtExecutionProvider', {'device_id': 0, 'trt_fp16_enable': True}), 
    'CUDAExecutionProvider'
]
yolo_session = ort.InferenceSession("yolov4_1_3_224_224_pacholek.onnx", providers=providers)
yolo_input_name = yolo_session.get_inputs()[0].name

# 4. UI


# Funkcja pomocnicza dla Road Following
def road_inference(frame):
    img = preprocess(frame).half() # Używamy Twojej funkcji preprocess i trybu FP16
    output = model_road(img).detach().cpu().numpy().flatten()
    return float(output[0])

print("Inicjalizacja zakończona ✔ (Modele Road i YOLO gotowe)")

Inicjalizacja zakończona ✔ (Modele Road i YOLO gotowe)


In [2]:
image_widget = widgets.Image(format='jpeg', width=224, height=224)
display(image_widget)

Image(value=b'', format='jpeg', height='224', width='224')

In [36]:
# ====== PARAMETRY JAZDY ======
SPEED           = 0.4
K_LANE          = 2
AVOID_GAIN      = 0.8      
DANGER_Y        = 0.1      # 0.25 było zbyt wysoko na obrazie (za daleko)
AREA_THRESHOLD  = 0.020    # Obniżone, by łatwiej "zaskoczyło"
DEAD_ZONE       = 0       # Mniejszy dead_zone = szybsza reakcja
STEERING_BIAS   = 0.03
CONF_LIMIT      = 0.85      # 0.90 to za dużo dla Jetsona w ruchu
AVOID_HOLD_TIME = 0.2      # 1.5s to za długo, auto nie wróci na tor      # Podtrzymanie uniku w sekundach    
AVOID_DECAY     = 0.5

print(f"Parametry załadowane! Próg wielkości (AREA): {AREA_THRESHOLD}")

Parametry załadowane! Próg wielkości (AREA): 0.02


In [38]:
# --- PARAMETRY KONFIGURACYJNE ---
MAX_SPEED = 0.45      # Prędkość na prostej
MIN_SPEED = 0.33     # Prędkość w ostrym skręcie / podczas uniku
# --------------------------------

active_avoidance = 0 

try:
    while True:
        frame = camera.value
        if frame is None: continue
        
        # 1. Obliczamy kierunek drogi (RFM)
        rfm_steer = road_inference(frame)
        lane_center_px = 112 + (rfm_steer * 112 * K_LANE)

        # 2. Detekcja przeszkód (YOLO)
        img_yolo = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB).transpose((2, 0, 1)).astype(np.float32) / 255.0
        blob = np.expand_dims(img_yolo, axis=0)
        yolo_outs = yolo_session.run(None, {yolo_input_name: blob})
        
        boxes = np.squeeze(yolo_outs[0])
        scores = np.squeeze(yolo_outs[1])
        if len(scores.shape) > 1: scores = scores[:, 0]
        
        current_frame_avoidance = 0
        best_conf = 0
        best_area = 0

        if len(scores) > 0:
            best_idx = np.argmax(scores)
            best_conf = scores[best_idx]
            
            if best_conf > CONF_LIMIT:
                box = boxes[best_idx]
                x1, y1, x2, y2 = (box * 224).astype(int)
                best_area = (box[2] - box[0]) * (box[3] - box[1])
                cx = (x1 + x2) / 2
                y_bottom = y2 / 224.0

                # --- NOWA LOGIKA: Filtr RFM-Y ---
                # Zakładamy, że RFM patrzy na horyzont drogi. 
                # Jeśli spód pachołka jest wyżej niż próg Danger lub zbyt daleko (wysoko), ignorujemy.
                # Dodatkowo sprawdzamy czy pachołek nie jest "za daleko" w stosunku do punktu RFM
                if y_bottom > DANGER_Y:
                    relative_dist = cx - lane_center_px
                    
                    if abs(relative_dist) > DEAD_ZONE:
                        weight = (y_bottom - DANGER_Y) / (1.0 - DANGER_Y)
                        current_frame_avoidance = -AVOID_GAIN * weight if relative_dist > 0 else AVOID_GAIN * weight

                # Rysowanie boxa i info
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(frame, f"C:{best_conf:.2f} A:{best_area:.3f}", (x1, y1-15), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)

        # --- LOGIKA DECAY ---
        if abs(current_frame_avoidance) > abs(active_avoidance):
            active_avoidance = current_frame_avoidance
        else:
            if active_avoidance > 0:
                active_avoidance = max(0, active_avoidance - AVOID_DECAY)
            elif active_avoidance < 0:
                active_avoidance = min(0, active_avoidance + AVOID_DECAY)

        # 3. Finalne sterowanie
        final_steering = np.clip(rfm_steer + active_avoidance + STEERING_BIAS, -1.0, 1.0)
        
        # --- NOWA LOGIKA: Prędkość zależna od skrętu ---
        # Im większy abs(final_steering), tym mniejsza prędkość.
        steering_impact = abs(final_steering)
        dynamic_speed = MAX_SPEED - (steering_impact * (MAX_SPEED - MIN_SPEED))
        dynamic_speed = np.clip(dynamic_speed, MIN_SPEED, MAX_SPEED)

        car.steering = -final_steering 
        car.throttle = -dynamic_speed

        # 4. Debug Video Overlay (z informacją o prędkości)
        cv2.rectangle(frame, (0, 0), (120, 80), (0, 0, 0), -1)
        cv2.putText(frame, f"CONF: {best_conf:.2f}", (5, 15), 1, 0.4, (255, 255, 255), 1)
        cv2.putText(frame, f"AREA: {best_area:.3f}", (5, 30), 1, 0.4, (255, 255, 255), 1)
        cv2.putText(frame, f"SPD:  {dynamic_speed:.2f}", (5, 45), 1, 0.4, (0, 255, 0), 1)
        cv2.putText(frame, f"AVD:  {active_avoidance:.2f}", (5, 60), 1, 0.4, (0, 255, 255), 1)

        cv2.line(frame, (int(lane_center_px), 0), (int(lane_center_px), 224), (0, 255, 0), 1)
        dy_px = int(DANGER_Y * 224)
        cv2.line(frame, (0, dy_px), (224, dy_px), (0, 255, 255), 2)
        
        image_widget.value = bgr8_to_jpeg(frame)
        print(f"\rSPD: {dynamic_speed:.2f} | ST: {-final_steering:.2f} | Avoid: {active_avoidance:.2f}", end="")

except KeyboardInterrupt:
    print("\nZatrzymano.")
finally:
    car.throttle = 0.0
    car.steering = STEERING_BIAS

SPD: 0.34 | ST: -0.88 | Avoid: 0.242
Zatrzymano.
